In [1]:
from docx import Document
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
import re


def clean_sheet_name(name: str) -> str:
    """
    Excel sheet name:
    - không quá 31 ký tự
    - không chứa: : \\ / ? * [ ]
    """
    name = re.sub(r'[:\\/*?\[\]]', '_', name)
    return name[:31]


def extract_tables_from_word_to_excel(word_path: str, excel_path: str) -> None:
    # Mở file Word
    doc = Document(word_path)

    # Tạo workbook Excel
    wb = Workbook()

    # Xóa sheet mặc định
    default_sheet = wb.active
    wb.remove(default_sheet)

    if not doc.tables:
        # Nếu không có bảng thì tạo 1 sheet thông báo
        ws = wb.create_sheet("NoTables")
        ws["A1"] = "Không tìm thấy bảng nào trong file Word."
    else:
        for table_index, table in enumerate(doc.tables, start=1):
            sheet_name = clean_sheet_name(f"Table_{table_index}")
            ws = wb.create_sheet(title=sheet_name)

            max_cols = 0

            for row_idx, row in enumerate(table.rows, start=1):
                row_values = []
                for cell in row.cells:
                    text = cell.text.strip()
                    row_values.append(text)

                max_cols = max(max_cols, len(row_values))

                for col_idx, value in enumerate(row_values, start=1):
                    ws.cell(row=row_idx, column=col_idx, value=value)

            # Tự động chỉnh độ rộng cột cơ bản
            for col_idx in range(1, max_cols + 1):
                max_length = 0
                col_letter = get_column_letter(col_idx)
                for cell in ws[col_letter]:
                    if cell.value is not None:
                        max_length = max(max_length, len(str(cell.value)))
                ws.column_dimensions[col_letter].width = min(max_length + 2, 50)

    # Lưu file Excel
    wb.save(excel_path)
    print(f"Đã lưu tất cả bảng từ '{word_path}' sang '{excel_path}'")


if __name__ == "__main__":
    word_file = r"D:\startup\muasamcong\BIDFinder\crawler_engine\raw_data\20260406\latest\IB2500326521_v00_778_QĐ-BVVH_Danh mục hàng hóa trúng thầu.docx"
    excel_file = r"D:\startup\muasamcong\BIDFinder\crawler_engine\raw_data\IB2500326521_v00_778_QĐ-BVVH_Danh mục hàng hóa trúng thầu.xlsx"
    extract_tables_from_word_to_excel(word_file, excel_file)

Đã lưu tất cả bảng từ 'D:\startup\muasamcong\BIDFinder\crawler_engine\raw_data\20260406\latest\IB2500326521_v00_778_QĐ-BVVH_Danh mục hàng hóa trúng thầu.docx' sang 'D:\startup\muasamcong\BIDFinder\crawler_engine\raw_data\IB2500326521_v00_778_QĐ-BVVH_Danh mục hàng hóa trúng thầu.xlsx'
